# 01. 이미지 분류 — CNN 직접 설계하기

이 노트북에서는 **합성곱 신경망(CNN)** 을 직접 설계해 이미지를 분류합니다.
딥러닝 컴퓨터비전의 가장 기본이 되는 실습으로, 다음 흐름을 따릅니다.

1. **데이터 준비** — CIFAR-10 데이터셋 로드 및 시각화
2. **모델 설계** — 합성곱 층으로 CNN을 직접 구성
3. **학습** — 손실 함수와 옵티마이저로 모델 학습
4. **평가** — 테스트 정확도 측정 및 결과 해석

> CIFAR-10은 10개 클래스(비행기, 자동차, 새, 고양이 등)의 32×32 컬러 이미지 6만 장으로 이루어진 표준 데이터셋입니다. 코드를 실행하면 `/workspace`에 자동으로 다운로드됩니다.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# 한글 폰트 (matplotlib에서 한글 깨짐 방지)
plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('사용 장치:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 1. 데이터 준비

CIFAR-10을 불러옵니다. 학습용 5만 장, 테스트용 1만 장으로 나뉘어 있습니다.

- `transforms.ToTensor()` : 이미지를 [0,1] 범위 텐서로 변환
- `transforms.Normalize(...)` : 채널별 평균·표준편차로 정규화 (학습 안정화)
- `DataLoader` : 데이터를 배치 단위로 묶어 공급

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

train_set = torchvision.datasets.CIFAR10(root='/workspace/data', train=True,
                                         download=True, transform=transform)
test_set = torchvision.datasets.CIFAR10(root='/workspace/data', train=False,
                                        download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=256, shuffle=False, num_workers=2)

classes = ('비행기', '자동차', '새', '고양이', '사슴', '개', '개구리', '말', '배', '트럭')
print('학습 데이터:', len(train_set), '장')
print('테스트 데이터:', len(test_set), '장')

### 데이터 살펴보기

학습이 잘 되려면 어떤 데이터를 다루는지 눈으로 확인하는 것이 중요합니다.
배치에서 몇 장을 꺼내 시각화합니다.

In [ ]:
def imshow(img):
    img = img * torch.tensor([0.2470, 0.2435, 0.2616]).view(3,1,1) \
        + torch.tensor([0.4914, 0.4822, 0.4465]).view(3,1,1)  # 정규화 복원
    return np.transpose(img.numpy(), (1, 2, 0)).clip(0, 1)

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(imshow(images[i]))
    ax.set_title(classes[labels[i]])
    ax.axis('off')
plt.suptitle('CIFAR-10 샘플 이미지', fontsize=14)
plt.tight_layout()
plt.show()

## 2. CNN 모델 설계

합성곱 신경망을 직접 설계합니다. 구조는 다음과 같습니다.

- **합성곱 블록 3개** : `Conv → BatchNorm → ReLU → MaxPool` 을 반복하며 채널을 32 → 64 → 128로 늘리고 공간 크기를 줄입니다.
- **완전연결 층** : 추출된 특징을 받아 10개 클래스로 분류합니다.

`BatchNorm`은 학습을 안정시키고, `Dropout`은 과적합을 줄입니다.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = SimpleCNN().to(device)
print(model)

## 3. 학습

- **손실 함수** : 분류 문제의 표준인 교차 엔트로피(`CrossEntropyLoss`)
- **옵티마이저** : Adam
- **에폭(epoch)** : 전체 데이터를 몇 번 반복 학습할지. 교육용으로 10으로 둡니다 (GPU에서 수 분).

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f'[{epoch+1:2d}/{EPOCHS}] 평균 손실: {running_loss/len(train_loader):.4f}')

print('학습 완료')

## 4. 평가

테스트 데이터로 정확도를 측정합니다. 직접 만든 CNN은 보통 **70% 안팎**의 정확도를 보입니다.

In [ ]:
model.eval()
correct = total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f'테스트 정확도: {100*correct/total:.2f}%')

### 정리

- 합성곱 층을 쌓아 이미지의 특징을 추출하고, 완전연결 층으로 분류하는 **CNN의 기본 구조**를 직접 구현했습니다.
- 직접 설계한 CNN으로 CIFAR-10에서 70% 안팎의 정확도를 얻었습니다.
- 다음 노트북(`02_transfer_learning.ipynb`)에서는 **사전학습된 모델(ResNet18)** 을 활용해 더 높은 정확도를 더 적은 학습으로 얻는 **전이학습**을 다룹니다.